# 00 Download: raw data for the Liver Cell Map

Downloads only the files the pipeline uses, into `DATA_ROOT/raw/` on the external drive
(see `config.py`). Safe to re-run: files already on disk with the right size are skipped.

| Dataset | What | Files |
|---|---|---|
| Xiong 2019, GEO GSE129516 | mouse liver, 3 healthy (chow) + 3 NASH | 6 CSVs, one per mouse |
| Ramachandran 2019, GEO GSE136103 | human liver, 5 healthy + 5 cirrhotic donors | 20 samples × (barcodes, genes, matrix) |
| MGI | mouse ↔ human gene pairs (orthologs) | `HOM_MouseHumanSequence.rpt` |

Not downloaded: the 4 blood samples and 2 mouse samples that also sit in GSE136103.

In [1]:
import os, urllib.parse, urllib.request, time
import pandas as pd
from config import *

def geo_url(gsm, fname):
    return GEO_SAMPLE_URL.format(prefix=gsm[:-3], gsm=gsm, fname=urllib.parse.quote(fname))

def series_sizes(gse):
    """Expected byte sizes from GEO's filelist.txt, used to verify each download."""
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{gse[:-3]}nnn/{gse}/suppl/filelist.txt"
    fl = pd.read_csv(url, sep="\t")
    return dict(zip(fl["Name"], fl["Size"]))

def fetch(url, dst, size=None, tries=3):
    if os.path.exists(dst) and (size is None or os.path.getsize(dst) == size):
        return "skip"
    for k in range(tries):
        try:
            tmp = dst + ".part"
            urllib.request.urlretrieve(url, tmp)
            if size is not None and os.path.getsize(tmp) != size:
                raise IOError(f"size {os.path.getsize(tmp)} != expected {size}")
            os.replace(tmp, dst)
            return "ok"
        except Exception as e:
            print(f"  retry {k + 1}/{tries}: {e}")
            time.sleep(3)
    raise RuntimeError(f"failed: {url}")

In [2]:
jobs = []
sizes = series_sizes("GSE129516")
for gsm, sample, _ in XIONG:
    f = XIONG_FILE.format(gsm=gsm, sample=sample)
    jobs.append((geo_url(gsm, f), os.path.join(RAW, "GSE129516", f), sizes.get(f)))
sizes = series_sizes("GSE136103")
for gsm, sample in RAMACHANDRAN:
    for pat in RAMACHANDRAN_FILES:
        f = pat.format(gsm=gsm, sample=sample)
        jobs.append((geo_url(gsm, f), os.path.join(RAW, "GSE136103", f), sizes.get(f)))
jobs.append((MGI_ORTHOLOGS_URL, os.path.join(RAW, "orthologs", "HOM_MouseHumanSequence.rpt"), None))

assert all(s is not None for _, d, s in jobs[:-1]), "a file name is missing from GEO's filelist"
print(len(jobs), "files,", round(sum(s for *_, s in jobs if s) / 1e6), "MB expected")

67 files, 347 MB expected


In [3]:
for url, dst, size in jobs:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(fetch(url, dst, size), os.path.relpath(dst, RAW))

ok GSE129516/GSM3714747_chow1_filtered_gene_bc_matrices.csv.gz


ok GSE129516/GSM3714748_chow2_filtered_gene_bc_matrices.csv.gz


ok GSE129516/GSM3714749_chow3_filtered_gene_bc_matrices.csv.gz


ok GSE129516/GSM3714750_nash1_filtered_gene_bc_matrices.csv.gz


ok GSE129516/GSM3714751_nash2_filtered_gene_bc_matrices.csv.gz


ok GSE129516/GSM3714752_nash3_filtered_gene_bc_matrices.csv.gz
ok GSE136103/GSM4041150_healthy1_cd45+_barcodes.tsv.gz


... (39 outputs left out)

ok GSE136103/GSM4041168_cirrhotic4_cd45+_matrix.mtx.gz
ok GSE136103/GSM4041169_cirrhotic5_cd45+_barcodes.tsv.gz


ok GSE136103/GSM4041169_cirrhotic5_cd45+_genes.tsv.gz


ok GSE136103/GSM4041169_cirrhotic5_cd45+_matrix.mtx.gz


ok orthologs/HOM_MouseHumanSequence.rpt


In [4]:
# Final check: every file present at the expected size
bad = [d for _, d, s in jobs if not os.path.exists(d) or (s and os.path.getsize(d) != s)]
print("all files OK" if not bad else f"PROBLEM with {len(bad)} files: {bad}")
assert not bad

all files OK
